In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score, precision_score, f1_score
from imblearn.over_sampling import SMOTE  
from imblearn.pipeline import Pipeline     
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import  RandomizedSearchCV,GridSearchCV
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

In [6]:
df = pd.read_csv('../../dataset/preprocessed/hotel_bookings.csv')

In [6]:
df.head()

,hotel,is_canceled,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,...,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,reservation_date,diff_reserved_room_type
0,Resort Hotel,0,342,0,0,2,0.0,0,BB,PRT,...,0,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01,2014-07-24,False
1,Resort Hotel,0,737,0,0,2,0.0,0,BB,PRT,...,0,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01,2013-06-24,False
2,Resort Hotel,0,7,0,1,1,0.0,0,BB,GBR,...,0,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01,2015-06-24,True
3,Resort Hotel,0,13,0,1,1,0.0,0,BB,GBR,...,0,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01,2015-06-18,False
4,Resort Hotel,0,14,0,2,2,0.0,0,BB,GBR,...,0,Transient,98.0,0,1,Check-Out,2015-07-03,2015-07-01,2015-06-17,False


In [7]:

string_cols = df.select_dtypes(include=['object']).columns
number_cols = df.select_dtypes(exclude=['object']).columns


df[string_cols] = df[string_cols].fillna('Unknown')
df[number_cols] = df[number_cols].fillna(0)


drop_cols = ['reservation_status', 'reservation_status_date', 'arrival_date', 'reservation_date']
X = df.drop(columns=[col for col in drop_cols if col in df.columns] + ['is_canceled'])
y = df['is_canceled']

X = pd.get_dummies(X, drop_first=True, dtype=float)

In [8]:
# 데이터 분리
X_train,X_test,y_train, y_test = train_test_split(X,y,
                                                  train_size=0.7,
                                                  random_state=1004)

In [9]:
print("모델 학습을 시작")

default_xgb = XGBClassifier(
    n_jobs=-1, 
    random_state=1004, 
    eval_metric='logloss'
)

# 모델 학습
default_xgb.fit(X_train, y_train)

모델 학습을 시작


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [10]:
y_pred = default_xgb.predict(X_test)


In [14]:
test_f1 = f1_score(y_test, y_pred)
test_recall = recall_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)

print(f"  Recall   : {test_recall:.2f}")
print(f"  Precision  : {test_precision:.2f}")
print(f"  F1-Score     : {test_f1:.2f}")



print("Classification Report")
print(classification_report(y_test, y_pred, target_names=['정상 투숙(0)', '예약 취소(1)']))

  Recall   : 0.79
  Precision  : 0.85
  F1-Score     : 0.82
Classification Report
              precision    recall  f1-score   support

    정상 투숙(0)       0.88      0.92      0.90     22390
    예약 취소(1)       0.85      0.79      0.82     13281

    accuracy                           0.87     35671
   macro avg       0.87      0.85      0.86     35671
weighted avg       0.87      0.87      0.87     35671



아래는 랜덤서치 틀

In [ ]:
# 타겟 지정, 날짜는 제외
drop_cols = ['reservation_status', 'reservation_status_date', 'arrival_date', 'reservation_date']       # 이건 데이터 나오는대로 바꿔줘야함
X = df.drop(columns=[col for col in drop_cols if col in df.columns] + ['is_canceled'])
y = df['is_canceled']

# 가변수화
X = pd.get_dummies(X, drop_first=True)



param_dist = {
    'classifier__n_estimators':[5],              # 나무의 개수
    'classifier__max_depth':[5],                    # 나무의 최대 깊이
    'classifier__learning_rate': [0.01, 0.05, 0.1, 0.2],       # 학습률 (보정 속도)
    'classifier__min_child_weight':[5],                # 가지치기를 결정하는 최소 관측치 가중치 합
    'classifier__subsample': [0.7, 0.8, 0.9, 1.0],            # 데이터 샘플링 비율
    'classifier__colsample_bytree': [0.7, 0.8, 0.9, 1.0],     # 피처 샘플링 비율
}



print("랜덤 서치 실행 중")

# SMOTE + XGBoost 파이프라인 구축
pipeline = Pipeline([
    ('smote', SMOTE(random_state=1004)),
    ('classifier', XGBClassifier(n_jobs=-1, random_state=1004, eval_metric='logloss'))
])

# k-fold = 5로 돌림
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1004)


random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=10, 
    scoring='f1',                # 3개의 지표를 시각화하긴 했지만, recall과 precision의 조화 평균이므로 f1 스코어링
    cv=cv,
    n_jobs=-1,
    random_state=1004,
    verbose=2                    # 튜닝 진행 상황을 화면에 출력
)


random_search.fit(X, y)




print("랜덤 서치 최적화 결과 성적표")
print(f"최상의 F1-Score 점수 : {random_search.best_score_:.3f}")
print("\n최종 채택된 베스트 하이퍼파라미터 조합:")

# 가독성을 위해 예쁘게 출력하기 위한 코드
best_params = random_search.best_params_
for param, value in best_params.items():
    clean_param = param.replace('classifier__', '')
    print(f" 🔹 {clean_param} : {value}")
print("="*40)

랜덤 서치 실행 중
Fitting 5 folds for each of 10 candidates, totalling 50 fits


In [ ]:
# 그리드서치
param_grid = {
    'classifier__n_estimators':[2] ,          
    'classifier__max_depth':[2],                
    'classifier__learning_rate': [0.05, 0.1],          
    'classifier__min_child_weight':[2],            
}



pipeline = Pipeline([
    ('smote', SMOTE(random_state=1004)),
    ('classifier', XGBClassifier(n_jobs=-1, random_state=1004, eval_metric='logloss'))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1004)

# GridSearchCV 정의
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,       
    scoring='f1',                # 평가 기준은 일단 f1 score 이게 조화 평균이기때문
    cv=cv,
    n_jobs=-1,
    verbose=2                    # 실시간 계산 진행 상황 출력
)


grid_search.fit(X_train, y_train)



print("\n" + "="*50)
print("최종 성적표")
print("="*50)

# 최적의 1등 모델 추출 및 전체 데이터 성능 평가
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X)

# 3대 지표 계산
final_f1 = f1_score(y, y_pred)
final_recall = recall_score(y, y_pred)
final_precision = precision_score(y, y_pred)

print(f"Grid 모델 성능")
print(f"Recall  : {final_recall:.4f}")
print(f"Precision  : {final_precision:.4f}")
print(f"F1-Score  : {final_f1:.4f}")
print("-"*50)

print("Classification Report")
print(classification_report(y, y_pred, target_names=['정상 투숙(0)', '예약 취소(1)']))
print("-"*50)

print("베스트 하이퍼파라미터")
best_params = grid_search.best_params_
for param, value in best_params.items():
    clean_param = param.replace('classifier__', '')
    print(f"  • {clean_param} : {value}")
print("="*50)

Fitting 5 folds for each of 2 candidates, totalling 10 fits


KeyboardInterrupt: 